<a href="https://colab.research.google.com/github/amosagekouassi-source/DI-Bootcamp/blob/main/Dailychallenge_J4_W6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-Tuning LLMs with LoRA
Ce notebook illustre le processus de Parameter-Efficient Fine-Tuning (PEFT) en utilisant LoRA sur un modèle Bloom.

In [1]:
%pip install peft datasets transformers accelerate --upgrade

In [2]:
import os
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

# Configuration du modèle et du tokenizer
model_name = "bigscience/bloomz-560m"
tokenizer = AutoTokenizer.from_pretrained(model_name)
foundation_model = AutoModelForCausalLM.from_pretrained(model_name)

# Chargement et préparation du dataset (10% sample)
data = load_dataset("Abirate/english_quotes", split="train")
data = data.train_test_split(test_size=0.1, seed=123)["test"] # On prend 10% pour l'exercice

def tokenize_function(samples):
    return tokenizer(samples["quote"], truncation=True, padding="max_length", max_length=128)

tokenized_data = data.map(tokenize_function, batched=True)
train_sample = tokenized_data.select(range(min(5, len(tokenized_data))))
print("Aperçu des données préparées :")
print(train_sample)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

Map:   0%|          | 0/251 [00:00<?, ? examples/s]

Aperçu des données préparées :
Dataset({
    features: ['quote', 'author', 'tags', 'input_ids', 'attention_mask'],
    num_rows: 5
})


In [6]:
# Mise à jour de torchao pour résoudre l'incompatibilité avec peft
%pip install "torchao>=0.16.0" --quiet

import peft
from peft import LoraConfig, get_peft_model

# Configuration de LoRA
lora_config = LoraConfig(
    r=1, # Rank
    lora_alpha=1,
    target_modules=["query_key_value"], # Spécifique à l'architecture BLOOM
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Application de LoRA au modèle de base
# Note: foundation_model doit être chargé au préalable
peft_model = get_peft_model(foundation_model, lora_config)
peft_model.print_trainable_parameters()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 26.1 MB/s eta 0:00:00


trainable params: 98,304 || all params: 559,312,896 || trainable%: 0.0176


In [7]:
import os
import transformers
from transformers import TrainingArguments, Trainer, AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
from datasets import load_dataset

# Re-chargement des composants essentiels si nécessaire
model_name = "bigscience/bloomz-560m"
if 'foundation_model' not in locals():
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    foundation_model = AutoModelForCausalLM.from_pretrained(model_name)

if 'train_sample' not in locals():
    data = load_dataset("Abirate/english_quotes", split="train")
    data = data.train_test_split(test_size=0.1, seed=123)["test"]
    tokenized_data = data.map(lambda x: tokenizer(x["quote"], truncation=True, padding="max_length", max_length=128), batched=True)
    train_sample = tokenized_data.select(range(min(5, len(tokenized_data))))

# Configuration de LoRA
lora_config = LoraConfig(
    r=1,
    lora_alpha=1,
    target_modules=["query_key_value"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

peft_model = get_peft_model(foundation_model, lora_config)

output_directory = "./peft_lab_outputs"
os.makedirs(output_directory, exist_ok=True)

training_args = TrainingArguments(
    report_to="none",
    output_dir=output_directory,
    per_device_train_batch_size=1,
    learning_rate=3e-2,
    num_train_epochs=1,
    use_cpu=True
)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_sample,
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

print("Début de l'entraînement...")
trainer.train()

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Début de l'entraînement...


Step,Training Loss


TrainOutput(global_step=5, training_loss=0.0, metrics={'train_runtime': 2714.7868, 'train_samples_per_second': 0.002, 'train_steps_per_second': 0.002, 'total_flos': 1161261219840.0, 'train_loss': 0.0, 'epoch': 1.0})

In [8]:
import time
import os

# Redéfinition par sécurité si la cellule précédente a échoué
output_directory = "./peft_lab_outputs"

# Sauvegarde du modèle
time_now = int(time.time())
peft_model_path = os.path.join(output_directory, f"peft_model_{time_now}")
trainer.model.save_pretrained(peft_model_path)

# Test d'inférence
inputs = tokenizer("Two things are infinite: ", return_tensors="pt").to(foundation_model.device)
outputs = peft_model.generate(
    input_ids=inputs["input_ids"],
    max_new_tokens=20,
    no_repeat_ngram_size=2
)

print("\nRésultat de la génération :")
print(tokenizer.batch_decode(outputs, skip_special_tokens=True))


Résultat de la génération :
['Two things are infinite: ']
